In [1]:
import sys
import pandas as pd
import logging
sys.path.append('/Users/BKieft/Metabolomics/metatlas2/metatlas2')
import database_interact as dbi
import pubchem_retrieval as pcr
import load_tools as ldt
import lcmsruns_tools as lrt
import ms1_ms2_analysis as msa
import rt_align_tools as rat
import targeted_analysis as tga
import targeted_gui as tgi
import logging_config as lcf

logger = lcf.setup_logging(log_level=logging.INFO)
pd.options.display.max_colwidth = 300

In [2]:
# Required: load config file
CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
# Required: table with compounds to add to db. Minimally, must have columns 'inchi_key' and 'label'
COMPOUND_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv", 
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_NEG.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_NEG.tsv"]
# Required: table of compounds; minimally, must have columns 'inchi_key', 'chromatography', 'polarity', 'rt_peak', 'mz', 'adduct'
ATLAS_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv",
                "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv"]
# Required: name of atlas (does not have to be unique) 
ATLAS_NAMES = ["HILIC Positive ISTD Default", "HILIC Positive QC Default"]
# Optional: description of atlas
ATLAS_DESC = [f"Default ISTD compounds for HILIC Positive", f"Default QC compounds for HILIC Positive"]
# Required: Project name
PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
# Required: paths based on config
PROJECT_FILES_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/lcmsruns"
PROJECT_DB_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/{PROJECT_NAME}.duckdb"
# Required: path to id report
ANALYSIS_OUTPUT_PATH = f'/Users/BKieft/Downloads/{PROJECT_NAME}'

In [ ]:
new_main_database = True
new_atlases = True
new_rt_alignment = True
new_analysis = True

In [4]:
if new_main_database is True:
    dbi.create_metatlas_database(CONFIG)
    for COMPOUNDS_INPUT in COMPOUND_INPUTS:
        compounds_df = ldt.load_compound_input(COMPOUNDS_INPUT)
        pcr.retrieve_pubchem_info(compounds_df, CONFIG)
        dbi.add_compounds_to_db(compounds_df, CONFIG, COMPOUNDS_INPUT)
    dbi.validate_database(CONFIG)

In [5]:
if new_atlases is True:
    new_atlas_info = ()
    for ATLAS_INPUT, ATLAS_NAME, ATLAS_DESC in zip(ATLAS_INPUTS, ATLAS_NAMES, ATLAS_DESC):
        atlas_compounds_df = ldt.load_atlas_input(ATLAS_INPUT)
        atlas_uid, atlas_name = dbi.create_atlas_from_compounds(atlas_compounds_df, ATLAS_NAME, ATLAS_DESC, CONFIG)
        new_atlas_info += ((atlas_uid, atlas_name),)
    dbi.validate_database(CONFIG)
    QC_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" in name)
    TARGET_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" not in name)
else:
    QC_ATLAS_UID = "atl-raw-7906f4bf395d4ed791f3763ed10f61c8"
    TARGET_ATLAS_UID = "atl-raw-2e24a9969a6a4758844b99162e9e0a03"
    logger.info(f"Skipping new atlas creation and using QC Atlas: {QC_ATLAS_UID} and TARGET Atlas: {TARGET_ATLAS_UID}")

2025-09-05 15:23:15 | metatlas2 | INFO | Skipping new atlas creation and using QC Atlas: atl-raw-7906f4bf395d4ed791f3763ed10f61c8 and TARGET Atlas: atl-raw-2e24a9969a6a4758844b99162e9e0a03


In [6]:
if new_rt_alignment is True:
    dbi.create_project_database(PROJECT_DB_PATH, CONFIG)
    lcmsrun_files = dbi.save_lcmsruns_to_db(PROJECT_DB_PATH, PROJECT_NAME, PROJECT_FILES_PATH, CONFIG)
    matches_df, matching_stats = msa.extract_and_match_qc_compounds(PROJECT_DB_PATH, QC_ATLAS_UID, CONFIG)
    best_model, model_results, model_stats = rat.build_rt_alignment_model(matches_df, CONFIG)
    ANALYSIS_ATLAS_UID = rat.apply_rt_correction_to_target(PROJECT_DB_PATH, TARGET_ATLAS_UID, CONFIG, best_model, lcmsrun_files, model_results)
else:
    ANALYSIS_ATLAS_UID = "atl-rta-0c89e582233e481ca4f1920611f6007f"
    logger.info(f"Skipping new RT alignment and using {ANALYSIS_ATLAS_UID}")

2025-09-05 15:23:15 | metatlas2 | INFO | Skipping new RT alignment and using atl-rta-0c89e582233e481ca4f1920611f6007f


In [7]:
if new_analysis is True:
    # Run targeted analysis workflow - now returns ProjectAnalysis object
    atlas_dataframe, project_analysis = tga.run_targeted_analysis_workflow(
        PROJECT_DB_PATH, ANALYSIS_ATLAS_UID, CONFIG
    )

    # Create GUI
    gui = tgi.create_gui(project_analysis, CONFIG, PROJECT_DB_PATH)
    gui._project_analysis = project_analysis  # Store ProjectAnalysis object for new workflow
    display(gui)

2025-09-05 15:23:15 | metatlas2.targeted_analysis | INFO | Setting up targeted analysis database...
2025-09-05 15:23:15 | metatlas2.targeted_analysis | INFO | Running fresh targeted analysis...
2025-09-05 15:23:15 | metatlas2.targeted_analysis | INFO | Loading target atlas...
2025-09-05 15:23:15 | metatlas2.database_interact | INFO | Retrieved 65 compounds from project database for atlas: HILIC Positive ISTD Default (RT Corrected) (atl-rta-0c89e582233e481ca4f1920611f6007f)
2025-09-05 15:23:15 | metatlas2.database_interact | INFO | RT-corrected atlas detected with experimental data
2025-09-05 15:23:15 | metatlas2.database_interact | INFO | Enriched 65 compounds with metadata from main database
2025-09-05 15:23:15 | metatlas2.targeted_analysis | INFO | Created Atlas dataframe with 65 compounds
2025-09-05 15:23:15 | metatlas2.simple_cache | INFO | Progress checkpoint saved: initialized at 20250905_152315
2025-09-05 15:23:15 | metatlas2.targeted_analysis | INFO | Loading experimental files

Processing files in parallel:   0%|          | 0/6 [00:00<?, ?it/s]

2025-09-05 15:23:19 | metatlas2.ms1_ms2_analysis | INFO |   Completed 1/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_22_Fall-RFa_7_Rg70to1050-CE205060norm-rhizo-InjBL-MeOH_Run123.h5 - 36 compounds
2025-09-05 15:23:19 | metatlas2.ms1_ms2_analysis | INFO |   Completed 2/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_28_Spring-VSp_2_Rg70to1050-CE102040norm-veg-core-ISTD_Run31.h5 - 49 compounds
2025-09-05 15:23:20 | metatlas2.ms1_ms2_analysis | INFO |   Completed 3/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_36_Fall-VFa_7_Rg70to1050-CE102040norm-veg-core-S1_Run136.h5 - 52 compounds
2025-09-05 15:23:20 | metatlas2.ms1_ms2_analysis | INFO |   Completed 4/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_48_Fall-BFa_7_Rg70to1050-CE102040norm-nonveg-core-S1_Run127.h5 - 57 compounds
2025-09-05 15:23:20 | metatlas2.ms1_ms2_analysis | INFO |   Completed 5/6: 20250707_JGI_MS_510172

2025-09-05 15:23:21 | metatlas2.targeted_gui | ERROR | Failed to save initial GUI state: [Errno 20] Not a directory: '/Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519.duckdb/cache/checkpoints/atl-rta-0c89e582233e481ca4f1920611f6007f'


<targeted_gui.create_gui.<locals>.GUIInterface at 0x16be97f50>

In [8]:
if 'project_analysis' in locals():
    sample_inchi_key = 'AGPKZVBTJJNPAG-WHFBIAKZSA-N'
    sample_compound = project_analysis.compounds[sample_inchi_key]
    print(sample_compound)
    print(f"Sample compound: {sample_compound.compound_name}")
    print(f"Original RT: {sample_compound.original_rt_peak}")
    print(f"Current RT: {sample_compound.rt_peak}")
    print(f"Is modified: {sample_compound.is_rt_modified or sample_compound.is_annotation_modified}")
    print(f"EIC files: {len(sample_compound.eic_data_files)}")
    print(f"MS2 files: {len(sample_compound.ms2_data_files.get('files', {}))}")

CompoundData(compound_uid='cmp-ccc9eb0009524b9e9bf5024b8b9f5864', inchi_key='AGPKZVBTJJNPAG-WHFBIAKZSA-N', compound_name='isoleucine (unlabeled)', formula='C6H13NO2', mz=132.1019, adduct='[M+H]+', polarity='positive', chromatography='HILICZ', mz_tolerance=5.0, original_rt_peak=9.636879252128477, original_rt_min=9.336856745099418, original_rt_max=9.937410039803753, rt_peak=9.636879252128477, rt_min=9.336856745099418, rt_max=9.937410039803753, ms1_notes='keep', ms2_notes='no selection', analyst_notes='', identification_notes='', best_eic_file='20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_9_Spring-RSp_6_Rg70to1050-CE102040norm-rhizo-S1_Run99.h5', best_eic_rt=9.498273849487305, best_eic_mz=132.10182189941406, best_eic_intensity=5603004.0, best_eic_ppm_error=0.11550780001071913, best_eic_rt_error=9.498273849487305, avg_eic_rt=9.508100668589273, avg_eic_intensity=1151822.8841145833, avg_eic_mz=132.10183461507162, best_ms2_file='20250707_JGI_MS_510172_SedRhizVegOut

In [9]:
if new_analysis is True:
    post_analysis_atlas_uid, targeted_analysis_uid, comprehensive_report = tga.run_post_analysis_workflow_v2(
        project_db_path=PROJECT_DB_PATH,
        analysis_atlas_uid=ANALYSIS_ATLAS_UID,
        project_analysis=gui._project_analysis,  # Use ProjectAnalysis object
        atlas_dataframe=atlas_dataframe,
        project_name=PROJECT_NAME,
        config=CONFIG,
        analysis_output_path=ANALYSIS_OUTPUT_PATH
    )
    
    logger.info("Class-based workflow completed successfully!")

# Print results
logger.info(f"Analysis saved with UID: {targeted_analysis_uid}")
logger.info(f"Post-analysis atlas UID: {post_analysis_atlas_uid}")
logger.info(f"Report saved to: {ANALYSIS_OUTPUT_PATH}")

2025-09-05 15:23:22 | metatlas2.targeted_analysis | INFO | Starting class-based post-analysis workflow...
2025-09-05 15:23:22 | metatlas2.targeted_analysis | INFO | Creating post-analysis atlas...
2025-09-05 15:23:22 | metatlas2.targeted_analysis | INFO | Creating post-analysis atlas with 1 modified compounds...
2025-09-05 15:23:22 | metatlas2.database_interact | INFO | Retrieved 65 compounds from project database for atlas: HILIC Positive ISTD Default (RT Corrected) (atl-rta-0c89e582233e481ca4f1920611f6007f)
2025-09-05 15:23:22 | metatlas2.database_interact | INFO | RT-corrected atlas detected with experimental data
2025-09-05 15:23:22 | metatlas2.database_interact | INFO | Enriched 65 compounds with metadata from main database
2025-09-05 15:23:22 | metatlas2.database_interact | INFO | Cloning atlas atl-rta-0c89e582233e481ca4f1920611f6007f with 1 modifications...
2025-09-05 15:23:22 | metatlas2.database_interact | INFO | Created new atlas: atl-tga-36a820424e284e2d8b547345d46c37d9
2025

Processing compounds:   0%|          | 0/65 [00:00<?, ?it/s]

2025-09-05 15:23:22 | metatlas2.database_interact | ERROR | Error saving Excel file: [Errno 21] Is a directory: '/Users/BKieft/Downloads/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519'
2025-09-05 15:23:22 | metatlas2.database_interact | ERROR | Error saving Excel file: [Errno 21] Is a directory: '/Users/BKieft/Downloads/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519'
2025-09-05 15:23:22 | metatlas2.targeted_analysis | INFO | Generated comprehensive report and saved to /Users/BKieft/Downloads/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519
2025-09-05 15:23:22 | metatlas2.targeted_analysis | INFO | Class-based post-analysis workflow completed successfully
2025-09-05 15:23:22 | metatlas2 | INFO | Class-based workflow completed successfully!
2025-09-05 15:23:22 | metatlas2 | INFO | Analysis saved with UID: tga-625b4c0349b647c488de6870e1392f32
2025-09-05 15:23:22 | metatlas2 | INFO | Post-analysis atlas UID: atl-tga-36a820424e284e2d8b5473

In [10]:
dbi.validate_database(CONFIG, PROJECT_DB_PATH)

2025-09-05 15:23:22 | metatlas2.database_interact | INFO | Database Validation:
2025-09-05 15:23:22 | metatlas2.database_interact | INFO |    Atlases: 9
2025-09-05 15:23:22 | metatlas2.database_interact | INFO |    Targeted analyses: 30
2025-09-05 15:23:22 | metatlas2.database_interact | INFO |    RT alignment models: 1
2025-09-05 15:23:22 | metatlas2.database_interact | INFO |    Experimental RT/MZ entries: 195
2025-09-05 15:23:22 | metatlas2.database_interact | INFO |    Available atlases:
2025-09-05 15:23:22 | metatlas2.database_interact | INFO |       atl-rta-0c89e582233e481ca4f1920611f6007f
2025-09-05 15:23:22 | metatlas2.database_interact | INFO |       atl-tga-5957c5649496448c9480b10032eb4247
2025-09-05 15:23:22 | metatlas2.database_interact | INFO |       atl-tga-d682e64f7145441593a2efd38e5ef480
2025-09-05 15:23:22 | metatlas2.database_interact | INFO |       atl-tga-a9e7fd0ab47f43c69e54d4248b5a4524
2025-09-05 15:23:22 | metatlas2.database_interact | INFO |       atl-tga-18863e